# Stage 1 — Non-Instruction Fine-Tuning
Domain: **Customer Support Assistant**. Adapts `Qwen2.5-1.5B-Instruct` to support-domain
language by training on raw response text (no instruction/response framing) before any
task-specific tuning happens in Stage 2.


In [2]:
# mount Drive and set the project path
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/Domain-SupportSpecialist'

# Sanity check: fail loudly and clearly if the project folder isn't where expected ---
REQUIRED_FILES = [
    'data/non_instruction_data.txt',
    'data/instruction_dataset.jsonl',
    'data/preference_dataset.jsonl',
]
missing = [p for p in REQUIRED_FILES if not os.path.exists(f'{PROJECT}/{p}')]
if missing:
    raise FileNotFoundError(
        "Missing from PROJECT: " + ", ".join(missing) +
        "\n\nThis almost always means the project folder is uploaded to the wrong place in Drive (e.g. nested inside another folder). Go to drive.google.com and confirm the path 'My Drive/Domain-SupportSpecialist/data/' contains these files directly -- move the folder to the top level of My Drive if it's nested, then Runtime -> Restart session and run this cell again."
    )

for sub in ['data', 'notebooks', 'reports', 'src',
            'adapters/stage1_non_instruction', 'adapters/stage2_sft', 'adapters/stage3_dpo']:
    os.makedirs(f'{PROJECT}/{sub}', exist_ok=True)

print('Project root OK:', PROJECT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root OK: /content/drive/MyDrive/Domain-SupportSpecialist


In [3]:
# fail fast if there's no GPU, with clear instructions
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected in this runtime.\n"
        "Fix: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, "
        "then Runtime -> Restart session, then run all cells again from the top.\n"
        "If T4 GPU won't attach at all / stays on 'None', you may be out of free GPU quota "
        "for today on this Google account -- try again later, or use a different account."
    )
print("GPU OK:", torch.cuda.get_device_name(0))

GPU OK: Tesla T4


In [4]:
# install Unsloth
# Install Unsloth. NOTE: do not pin an old trl version here -- unsloth_zoo needs a
# modern trl, and forcing an old one causes both import errors and a
# 'Trainer.__init__() got an unexpected keyword argument tokenizer' crash later
# (modern trl/transformers renamed tokenizer= to processing_class=).
!pip install --no-deps unsloth
!pip install unsloth_zoo
!pip install --no-deps peft accelerate bitsandbytes xformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 120.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Succes

In [5]:
# confirm GPU is attached
!nvidia-smi

Sat Jul 11 10:11:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# load the base model in 4-bit (QLoRA)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RM

In [7]:
# the 10 fixed evaluation questions + a small ask() helper.
# Reused unchanged across Stages 1, 2, and 3 so every comparison is apples-to-apples.
QUESTIONS = [
    "How can I cancel my order after it has been placed?",
    "My package says delivered but I never received it, what do I do?",
    "How long does a refund take to appear on my card?",
    "Can I change the delivery address after checkout?",
    "The item I received is damaged, what are my options?",
    "How do I track my order status?",
    "I was charged twice for one order, how do I fix this?",
    "Can I get a replacement instead of a refund?",
    "What happens if I miss the delivery attempt?",
    "How do I apply a discount code after placing an order?",
]

def ask(prompt):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True,
                add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

base_answers = {q: ask(q) for q in QUESTIONS}
for q, a in base_answers.items():
    print("Q:", q, "\nA:", a, "\n---")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Q: How can I cancel my order after it has been placed? 
A: To cancel an order that you have already placed, follow these steps:

1. **Check Your Order Status**: First, check your order status to see if the order is still in progress or if it's completed. This information will help determine whether cancellation is possible.

2. **Contact Customer Support**: If the order is not yet processed (completed), contact the customer support of the company from which you ordered. Provide them with your order details and ask for assistance on how to proceed with cancelling the order.

3. **Use Order Cancellation Features**: Some companies provide specific features within their website or app where customers can directly cancel orders. Check if there are any such options available and use them accordingly.

4. **Review Terms and Conditions**: Before proceeding, always review the terms and conditions provided when placing the order. Understanding what is allowed under those rules helps ensure that 

In [8]:
# write the base-model evaluation report
rows_md = "\n".join(
    f"| {q} | {base_answers[q].strip()} | _(fill in the problem you see)_ |"
    for q in QUESTIONS
)
report = f"""# Base Model Evaluation -- Qwen2.5-1.5B-Instruct (before any fine-tuning)

| Question | Base Model Answer | Problem |
|---|---|---|
{rows_md}
"""
with open(f'{PROJECT}/reports/base_model_evaluation.md', 'w', encoding='utf-8') as f:
    f.write(report)


## Load the raw domain text and chunk it

In [9]:
# load non_instruction_data.txt (already in the repo under data/)
with open(f'{PROJECT}/data/non_instruction_data.txt') as f:
    raw = f.read()

paragraphs = [p.strip() for p in raw.split("\n\n") if p.strip()]
from datasets import Dataset
raw_ds = Dataset.from_dict({"text": paragraphs})
print(raw_ds)
print(raw_ds[0]["text"][:300])

Dataset({
    features: ['text'],
    num_rows: 184
})
I get it your need to cancel your purchase with the order number #SC-28089. Let me guide you through the process:


## Attach LoRA and train on raw text
This is a short warm-up run: the goal is domain vocabulary and tone, not task-following
(that's Stage 2). Keep `max_steps` low so it finishes in a few minutes.

In [10]:
#LoRA + train
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, lora_alpha = 16, lora_dropout = 0,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
)

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model, processing_class = tokenizer, train_dataset = raw_ds,
    dataset_text_field = "text", max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 60,   # short on purpose: a warm-up stage
        learning_rate = 2e-4, fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(), logging_steps = 5,
        output_dir = "stage1_out", optim = "adamw_8bit",
        save_strategy = "no",
    ),
)
trainer.train()

Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/184 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 184 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,1.939523
10,2.025378
15,1.705911
20,1.527913
25,1.488091
30,1.312090
35,1.178858
40,1.151445
45,1.196575
50,0.952569


TrainOutput(global_step=60, training_loss=1.3765993277231852, metrics={'train_runtime': 126.663, 'train_samples_per_second': 3.79, 'train_steps_per_second': 0.474, 'total_flos': 369875634020352.0, 'train_loss': 1.3765993277231852, 'epoch': 2.608695652173913})

In [11]:
# save the adapter and sanity-check generation
import os
os.makedirs(f"{PROJECT}/adapters/stage1_non_instruction", exist_ok=True)
model.save_pretrained(f"{PROJECT}/adapters/stage1_non_instruction")
tokenizer.save_pretrained(f"{PROJECT}/adapters/stage1_non_instruction")

FastLanguageModel.for_inference(model)
print(ask("Tell me about the returns process."))

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Domain-SupportSpecialist/adapters/stage1_non_instruction/tokenizer_config.json.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in

The returns process is designed to help you in obtaining your items if they don't meet your expectations or aren't as described. Here's an overview of our typical returns process: 

1. Review Policy: Before initiating any return, please review our policy which outlines the conditions under which returns can be made.

2. Notify Us: Once you decide that a return is necessary, notify us via email at support@example.com with the reason for the return and provide relevant details such as order number #SC-0356789.

3. Request Refund: Provide us with all necessary documents like purchase confirmation and any evidence supporting your case for a refund. This will enable us to investigate further and make a decision on whether to grant your request.

4. Check Eligibility: We'll carefully examine your case and determine if it meets our criteria for refunds. If approved, we'll initiate the refund processing accordingly.

Please note that specific policies may vary based on the nature of your purch